# Chapter 12 — Problem Set 1: Solutions

Runnable solutions with explanations. All exercises run **offline** with `MockLLMClient`.

---
*Generated by Berta AI*

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))

from prompt_templates import PromptTemplate, FewShotTemplate, FewShotExample
from llm_clients import MockLLMClient
from evaluation_utils import safe_json_parse

import json
client = MockLLMClient()

## 1. Rewrite a Vague Prompt — Solution

Vague: "Tell me about this review". Better prompt has an **explicit task**, **output format**, and an **example**.

In [ ]:
rewritten = PromptTemplate(
    name='review_v1',
    system='You analyze short product reviews. Be precise and concise.',
    template=(
        "Task: classify the sentiment as one of positive, negative, neutral.\n"
        "Return only the label.\n\n"
        "Example:\nReview: 'I love it.' -> positive\n\n"
        "Review: '{{ review }}'\nLabel:"
    ),
)
out = client.complete(rewritten.render(review='The phone is fast but the battery dies quickly.'))
print(out.text)

## 2. Few-Shot Examples — Solution

In [ ]:
few = FewShotTemplate(
    name='topic_few_shot',
    template='',
    examples=[
        FewShotExample('New phone announced today.', 'tech'),
        FewShotExample('Election results came in late.', 'politics'),
        FewShotExample('Star quarterback returns.', 'sports'),
        FewShotExample('Award-winning film opens this week.', 'entertainment'),
    ],
)
for h in ['Stock market closes higher on tech rally', 'Olympic team announces new coach']:
    p = few.render(input=h, instruction='Classify into tech/sports/politics/entertainment.')
    print(f'{h!r} -> {client.complete(p).text!r}')

## 3. Structured-Output Schema — Solution

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import List

class ReviewExtract(BaseModel):
    product_name: str
    rating: int = Field(ge=1, le=5)
    pros: List[str] = []
    cons: List[str] = []

schema = json.dumps(ReviewExtract.model_json_schema(), indent=2)
tmpl = PromptTemplate(
    name='review_extract',
    template='Return JSON matching this schema:\n```json\n' + schema + '\n```\n\nReview: {{ text }}\nJSON:',
)
review = 'The Foo Phone X is fast and bright (4 stars), but battery life is poor.'
raw = client.complete(tmpl.render(text=review)).text
print('Raw:', raw)

parsed = safe_json_parse(raw) or {}
# Mock won't return a perfect schema match; demonstrate fallback construction
fallback = ReviewExtract(product_name='Foo Phone X', rating=4, pros=['fast', 'bright'], cons=['battery life is poor'])
try:
    obj = ReviewExtract.model_validate(parsed)
except ValidationError:
    obj = fallback
print('Parsed object:', obj)

## 4. Tricky Example — Solution

In [ ]:
prompt = PromptTemplate(name='cls', template='Sentiment (positive/negative/neutral): {{ text }}\nLabel:')
text = 'The food was amazing but the service was awful.'
print('Mock label:', client.complete(prompt.render(text=text)).text)
print('\nThis is hard for single-label classifiers because the review contains both strongly positive and strongly negative aspects ("food=amazing", "service=awful"); aspect-based sentiment analysis would split it into per-aspect labels rather than forcing a single overall label.')

## 5. Token Counting — Solution

In [ ]:
def approx_token_count(text: str) -> int:
    return max(1, len(text) // 4)

long_prompt = 'Once upon a time, ' * 200
print('Heuristic tokens:', approx_token_count(long_prompt))

resp = client.complete(long_prompt)
print('Mock prompt_tokens:', resp.prompt_tokens, '(both use the same heuristic)')

## 6. Parse JSON Safely — Solution

In [ ]:
from pydantic import BaseModel
from typing import Optional, Type

class Sent(BaseModel):
    label: str = 'neutral'
    confidence: float = 0.0

def parse_or_default(raw: str, schema_cls: Type[BaseModel]) -> BaseModel:
    parsed = safe_json_parse(raw)
    if parsed is None:
        return schema_cls()
    try:
        return schema_cls.model_validate(parsed)
    except Exception:
        return schema_cls()

raw_ok = 'Here is your answer: {"label": "positive", "confidence": 0.9}'
raw_bad = 'sorry, I cannot do that'
print(parse_or_default(raw_ok, Sent))
print(parse_or_default(raw_bad, Sent))

---
*Generated by Berta AI*